> **Note**: This notebook performs targeted data augmentation for the general Waste Detection YOLO pipeline. Class names are read dynamically from `data.yaml`. Minority classes receive extra augmentation.

# 🎨 Waste Detection — Data Augmentation (Notebook 05)

### Overview
Augments the cleaned YOLO dataset to improve model generalization and address class imbalance.

### Strategy
- **Minority classes** receive more augmentation passes (targeted oversampling)
- **Majority classes** receive standard augmentation
- **Val/Test sets** are copied without augmentation (for fair evaluation)
- Albumentations transforms preserve YOLO bounding box coordinates

### Pipeline Position
```
NB 04 (Cleaning) → [taco_yolo_cleaned/] → NB 05 (THIS) → [taco_yolo_augmented/] → NB 06
```

## 1. Environment Setup & Library Imports

In [ ]:
!pip install -q albumentations rich tqdm pyyaml pandas opencv-python matplotlib pillow

In [ ]:
import os
import shutil
import json
import yaml
import random
import math
from pathlib import Path
from typing import List, Dict, Tuple, Any, Set
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import albumentations as A
from tqdm.auto import tqdm
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()
random.seed(42)
np.random.seed(42)

## 2. Load Configuration and Input Dataset
Load the cleaned dataset from Notebook 04.

In [ ]:
# ==========================================
# Configuration — Adjust paths as needed
# ==========================================
PROJECT_ROOT = Path('/content/drive/MyDrive/PlasticSense_AI')

# Input: Output of NB 04 (Cleaned Dataset)
INPUT_YOLO_DIR = PROJECT_ROOT / 'datasets/taco_yolo_cleaned'
YAML_PATH = INPUT_YOLO_DIR / 'data.yaml'

# Output: Augmented dataset
OUTPUT_YOLO_DIR = PROJECT_ROOT / 'datasets/taco_yolo_augmented'
REPORTS_DIR = OUTPUT_YOLO_DIR / 'reports'

# ==========================================
# Load data.yaml
# ==========================================
if not YAML_PATH.exists():
    raise FileNotFoundError(f"{YAML_PATH} not found. Run Notebook 04 first.")

with open(YAML_PATH, 'r') as f:
    config = yaml.safe_load(f)

CLASS_NAMES = {int(k): v for k, v in config.get('names', {}).items()}
NUM_CLASSES = len(CLASS_NAMES)

console.print(f"[green]✔ Loaded data.yaml — {NUM_CLASSES} classes[/green]")
console.print(f"[cyan]  Input:  {INPUT_YOLO_DIR}[/cyan]")
console.print(f"[cyan]  Output: {OUTPUT_YOLO_DIR}[/cyan]")

## 3. Analyze Class Balance for Targeted Augmentation
Calculate per-class frequencies to determine augmentation multipliers.

In [ ]:
# ==========================================
# Analyze class balance in training set
# ==========================================
def analyze_class_balance(yolo_dir: Path, class_names: dict) -> Tuple[dict, dict]:
    """Count objects per class and map images to their classes."""
    class_counts = {cid: 0 for cid in class_names.keys()}
    img_classes = {}  # img_path → set of class IDs

    lbl_dir = yolo_dir / 'labels' / 'train'
    img_dir = yolo_dir / 'images' / 'train'

    if not lbl_dir.exists():
        console.print("[red]Training labels directory not found![/red]")
        return class_counts, img_classes

    for lbl_path in lbl_dir.glob('*.txt'):
        # Find corresponding image
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG']:
            candidate = img_dir / f"{lbl_path.stem}{ext}"
            if candidate.exists():
                img_path = candidate
                break

        if img_path is None:
            continue

        classes_in_img = set()
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls_id = int(parts[0])
                    if cls_id in class_counts:
                        class_counts[cls_id] += 1
                        classes_in_img.add(cls_id)

        if classes_in_img:
            img_classes[img_path] = classes_in_img

    return class_counts, img_classes

class_counts, img_classes = analyze_class_balance(INPUT_YOLO_DIR, CLASS_NAMES)

# Calculate augmentation multipliers
max_count = max(class_counts.values()) if class_counts else 1
median_count = int(np.median([v for v in class_counts.values() if v > 0])) if any(v > 0 for v in class_counts.values()) else 1

aug_multipliers = {}
for cid, count in class_counts.items():
    if count == 0:
        aug_multipliers[cid] = 0  # Can't augment what doesn't exist
    elif count < median_count * 0.3:
        aug_multipliers[cid] = 5  # Very rare → heavy augmentation
    elif count < median_count * 0.7:
        aug_multipliers[cid] = 3  # Underrepresented → moderate
    elif count < median_count:
        aug_multipliers[cid] = 2  # Slightly below median
    else:
        aug_multipliers[cid] = 1  # At or above median → standard

# Display
table = Table(title='Class Balance & Augmentation Plan', show_header=True, show_lines=False)
table.add_column('ID', style='cyan', justify='right')
table.add_column('Class', style='white')
table.add_column('Count', justify='right')
table.add_column('Multiplier', justify='right')

for cid in sorted(class_counts.keys()):
    count = class_counts[cid]
    mult = aug_multipliers[cid]
    style = 'red' if mult >= 3 else ('yellow' if mult == 2 else 'green')
    table.add_row(str(cid), CLASS_NAMES[cid], str(count), f'[{style}]x{mult}[/{style}]')

console.print(table)
console.print(f"[cyan]Median class count: {median_count}[/cyan]")

## 4. Albumentations Pipeline
Robust augmentation transforms that handle real-world conditions: lighting, weather, perspective, occlusion.

In [ ]:
# ==========================================
# Augmentation pipeline — realistic transforms
# ==========================================
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.1),
    A.Rotate(limit=15, p=0.3, border_mode=cv2.BORDER_REFLECT_101),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.CLAHE(clip_limit=2.0, p=0.2),
    A.HueSaturationValue(p=0.2),
    A.RGBShift(p=0.2),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.MotionBlur(p=0.15),
    A.GaussNoise(std_dev=(10.0, 30.0), p=0.2),
    A.RandomGamma(p=0.2),
    A.RandomShadow(shadow_roi=(0, 0.5, 1, 1), num_shadows_limit=(1, 2), shadow_dimension=3, p=0.15),
    A.RandomFog(fog_coef_range=(0.1, 0.3), p=0.05),
    A.RandomRain(brightness_coefficient=0.9, drop_length=20, p=0.03),
    A.Perspective(scale=(0.02, 0.05), p=0.1, border_mode=cv2.BORDER_REFLECT_101),
    A.Sharpen(p=0.15),
    A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(1, 8), hole_width_range=(1, 8), p=0.1),
    A.ImageCompression(quality_range=(80, 100), p=0.15)
], bbox_params=A.BboxParams(
    format='yolo',
    label_fields=['class_labels'],
    min_visibility=0.4,
    min_area=100
))

console.print("[green]✔ Augmentation pipeline configured[/green]")

## 5. Execute Augmentation
Augment training images based on class-specific multipliers. Val/Test are copied without augmentation.

In [ ]:
# ==========================================
# Augmentation execution
# ==========================================
def augment_image(img_path: Path, lbl_path: Path, output_img_dir: Path, output_lbl_dir: Path,
                  aug_id: int, transform: A.Compose) -> bool:
    """Apply augmentation to a single image and its labels."""
    img = cv2.imread(str(img_path))
    if img is None:
        return False
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Read YOLO labels
    bboxes = []
    class_labels = []
    if lbl_path.exists():
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls_id = int(parts[0])
                    x_c, y_c, w, h = [float(p) for p in parts[1:]]
                    bboxes.append([x_c, y_c, w, h])
                    class_labels.append(cls_id)

    if not bboxes:
        return False

    try:
        transformed = transform(image=img, bboxes=bboxes, class_labels=class_labels)
    except Exception:
        return False

    aug_img = transformed['image']
    aug_bboxes = transformed['bboxes']
    aug_labels = transformed['class_labels']

    if not aug_bboxes:
        return False

    # Save augmented image
    aug_name = f"{img_path.stem}_aug{aug_id}"
    aug_img_bgr = cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(output_img_dir / f"{aug_name}.jpg"), aug_img_bgr)

    # Save augmented labels
    with open(output_lbl_dir / f"{aug_name}.txt", 'w') as f:
        for bbox, cls_id in zip(aug_bboxes, aug_labels):
            x_c, y_c, w, h = bbox
            f.write(f"{cls_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")

    return True

# ==========================================
# Main augmentation loop
# ==========================================
# Clear output
if OUTPUT_YOLO_DIR.exists():
    shutil.rmtree(OUTPUT_YOLO_DIR)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Copy val and test without augmentation
for split in ['val', 'test']:
    src_img = INPUT_YOLO_DIR / 'images' / split
    src_lbl = INPUT_YOLO_DIR / 'labels' / split
    dst_img = OUTPUT_YOLO_DIR / 'images' / split
    dst_lbl = OUTPUT_YOLO_DIR / 'labels' / split

    if src_img.exists():
        shutil.copytree(str(src_img), str(dst_img))
    if src_lbl.exists():
        shutil.copytree(str(src_lbl), str(dst_lbl))

    count = len(list(dst_img.glob('*'))) if dst_img.exists() else 0
    console.print(f"[green]  ✔ {split}: {count} images copied (no augmentation)[/green]")

# Augment training set
train_img_dir = OUTPUT_YOLO_DIR / 'images' / 'train'
train_lbl_dir = OUTPUT_YOLO_DIR / 'labels' / 'train'
train_img_dir.mkdir(parents=True, exist_ok=True)
train_lbl_dir.mkdir(parents=True, exist_ok=True)

# First copy all original training images
src_train_img = INPUT_YOLO_DIR / 'images' / 'train'
src_train_lbl = INPUT_YOLO_DIR / 'labels' / 'train'

original_count = 0
for img_path in sorted(src_train_img.glob('*')):
    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
        shutil.copy2(str(img_path), str(train_img_dir / img_path.name))
        lbl_path = src_train_lbl / f"{img_path.stem}.txt"
        if lbl_path.exists():
            shutil.copy2(str(lbl_path), str(train_lbl_dir / lbl_path.name))
        original_count += 1

console.print(f"[green]  ✔ train: {original_count} original images copied[/green]")

# Now augment based on class multipliers
aug_count = 0
for img_path, classes_in_img in tqdm(img_classes.items(), desc="Augmenting"):
    # Determine max multiplier for this image
    max_mult = max(aug_multipliers.get(cid, 1) for cid in classes_in_img)

    lbl_path = INPUT_YOLO_DIR / 'labels' / 'train' / f"{img_path.stem}.txt"

    for aug_id in range(1, max_mult):  # Already have original (mult=1)
        success = augment_image(img_path, lbl_path, train_img_dir, train_lbl_dir, aug_id, transform)
        if success:
            aug_count += 1

console.print(f"[green]  ✔ Generated {aug_count} augmented training images[/green]")

final_train_count = len(list(train_img_dir.glob('*')))
console.print(Panel.fit(
    f"[bold green]Augmentation Complete[/bold green]\n\n"
    f"Original train images: {original_count}\n"
    f"Augmented images added: {aug_count}\n"
    f"Total train images: {final_train_count}"
))

## 6. Generate Augmented `data.yaml`

In [ ]:
# ==========================================
# Generate data.yaml for augmented dataset
# ==========================================
aug_yaml = {
    'path': str(OUTPUT_YOLO_DIR.absolute()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': NUM_CLASSES,
    'names': CLASS_NAMES
}

aug_yaml_path = OUTPUT_YOLO_DIR / 'data.yaml'
with open(aug_yaml_path, 'w') as f:
    yaml.dump(aug_yaml, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

console.print(f"[green]✔ Generated data.yaml for augmented dataset[/green]")
console.print(f"[cyan]  nc: {NUM_CLASSES}, path: {aug_yaml_path}[/cyan]")

## 7. Augmentation Visualization
Compare original and augmented samples side-by-side.

In [ ]:
# Show some augmented examples
aug_imgs = [p for p in train_img_dir.glob('*_aug*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
if aug_imgs:
    samples = random.sample(aug_imgs, min(8, len(aug_imgs)))
    cols = 4
    rows = math.ceil(len(samples) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows))
    if rows * cols == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for i, img_path in enumerate(samples):
        img = cv2.imread(str(img_path))
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[i].imshow(img)
            axes[i].set_title(img_path.name, fontsize=8)
        axes[i].axis('off')

    for j in range(len(samples), len(axes)):
        axes[j].axis('off')

    plt.suptitle('Augmented Training Samples', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    console.print("[yellow]No augmented images to display.[/yellow]")

## 8. Augmentation Reports

In [ ]:
# ==========================================
# Generate reports
# ==========================================
aug_report = {
    'original_train_images': original_count,
    'augmented_images_added': aug_count,
    'total_train_images': final_train_count,
    'val_images': len(list((OUTPUT_YOLO_DIR / 'images' / 'val').glob('*'))) if (OUTPUT_YOLO_DIR / 'images' / 'val').exists() else 0,
    'test_images': len(list((OUTPUT_YOLO_DIR / 'images' / 'test').glob('*'))) if (OUTPUT_YOLO_DIR / 'images' / 'test').exists() else 0,
    'num_classes': NUM_CLASSES,
    'augmentation_multipliers': {CLASS_NAMES[k]: v for k, v in aug_multipliers.items()}
}

with open(REPORTS_DIR / 'augmentation_report.json', 'w') as f:
    json.dump(aug_report, f, indent=4)

pd.DataFrame([{k: v for k, v in aug_report.items() if k != 'augmentation_multipliers'}]).to_csv(
    REPORTS_DIR / 'augmentation_report.csv', index=False
)

console.print(f"[green]✔ Reports saved to {REPORTS_DIR}[/green]")

## 9. Final Summary

In [ ]:
console.print(Panel.fit(
    f"[bold green]Data Augmentation Complete[/bold green]\n\n"
    f"[bold cyan]✔ Classes:[/bold cyan] {NUM_CLASSES}\n"
    f"[bold cyan]✔ Train:[/bold cyan] {final_train_count} images ({original_count} original + {aug_count} augmented)\n"
    f"[bold cyan]✔ Val/Test:[/bold cyan] Copied without augmentation\n\n"
    f"[bold]Output:[/bold] {OUTPUT_YOLO_DIR}\n\n"
    f"[bold magenta]Next Notebook:[/bold magenta] 06_YOLOv11_Training.ipynb"
))